# Colab Setup

## Objective
Mount Google Drive, extract nested zip files, and locate dataset CSVs automatically.

### Tasks
- Mount Google Drive
- Set path to the main zip file
- Extract outer zip
- Detect and extract any inner zips
- Locate train.csv and test.csv
- Verify files are ready for tokenization

## Step 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2 — Configure Paths

فقط این دو متغیر رو با مسیر واقعی فایلت عوض کن.

In [2]:
import os
import zipfile

DRIVE_ZIP_PATH = "/content/drive/MyDrive/jigsaw-toxic-comment-classification-challenge.zip"

EXTRACT_DIR = "/content/drive/MyDrive/jigsaw-data/"

os.makedirs(EXTRACT_DIR, exist_ok=True)
print(f"Extract directory ready: {EXTRACT_DIR}")

Extract directory ready: /content/drive/MyDrive/jigsaw-data/


## Step 3 — Extract Nested Zips Automatically

این تابع زیپ اصلی رو باز می‌کنه، سپس هر زیپ داخلی رو هم به صورت خودکار استخراج می‌کنه.

In [3]:
def extract_nested_zips(zip_path, extract_to):
    """Recursively extract all zip files found inside a zip."""
    print(f"Extracting: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_to)

    # find any inner zips that were just extracted
    for root, dirs, files in os.walk(extract_to):
        for filename in files:
            if filename.endswith(".zip"):
                inner_zip = os.path.join(root, filename)
                inner_dir = os.path.join(root, filename.replace(".zip", ""))
                os.makedirs(inner_dir, exist_ok=True)
                extract_nested_zips(inner_zip, inner_dir)  # recursive call
                os.remove(inner_zip)  # remove inner zip after extraction

extract_nested_zips(DRIVE_ZIP_PATH, EXTRACT_DIR)
print("\nAll zips extracted successfully.")

Extracting: /content/drive/MyDrive/jigsaw-toxic-comment-classification-challenge.zip
Extracting: /content/drive/MyDrive/jigsaw-data/sample_submission.csv.zip
Extracting: /content/drive/MyDrive/jigsaw-data/test.csv.zip
Extracting: /content/drive/MyDrive/jigsaw-data/test_labels.csv.zip
Extracting: /content/drive/MyDrive/jigsaw-data/train.csv.zip

All zips extracted successfully.


## Step 4 — Locate CSV Files

پیدا کردن خودکار مسیر `train.csv` و `test.csv` در هر جایی از پوشه‌ها.

In [4]:
TRAIN_PATH = None
TEST_PATH  = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    for filename in files:
        if filename == "train.csv":
            TRAIN_PATH = os.path.join(root, filename)
        if filename == "test.csv":
            TEST_PATH = os.path.join(root, filename)

assert TRAIN_PATH, "train.csv not found — check your zip structure"
assert TEST_PATH,  "test.csv not found — check your zip structure"

print(f"train.csv  →  {TRAIN_PATH}")
print(f"test.csv   →  {TEST_PATH}")

train.csv  →  /content/drive/MyDrive/jigsaw-data/train.csv/train.csv
test.csv   →  /content/drive/MyDrive/jigsaw-data/test.csv/test.csv


## Step 5 — Quick Sanity Check

In [5]:
import pandas as pd

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f"train shape: {train.shape}")
print(f"test shape : {test.shape}")
print("Dataset is ready.")

train shape: (159571, 8)
test shape : (153164, 2)
Dataset is ready.


## Step 6 — Export Paths for Other Notebooks

این مسیرها رو در نوت‌بوک‌های بعدی (`tokenization.ipynb` و `bilstm.ipynb`) جایگزین مسیرهای قدیمی کن.

In [6]:
print("Use these paths in tokenization.ipynb:")
print(f"  TRAIN_PATH = '{TRAIN_PATH}'")
print(f"  TEST_PATH  = '{TEST_PATH}'")

Use these paths in tokenization.ipynb:
  TRAIN_PATH = '/content/drive/MyDrive/jigsaw-data/train.csv/train.csv'
  TEST_PATH  = '/content/drive/MyDrive/jigsaw-data/test.csv/test.csv'


## Key Findings

- Drive successfully mounted and zip extracted recursively.
- `train.csv` and `test.csv` located automatically regardless of folder depth.
- Dataset dimensions verified and ready.

### Next Steps

- Copy the printed paths into `tokenization.ipynb`
- Run `tokenization.ipynb` to generate `X_train.npy`, `X_test.npy`, `y_train.npy`, `tokenizer.pkl`
- Run `bilstm.ipynb` to train the model